# 7.    UTS1

# 1.	Lakukan analisa klasifikasikan berita dengan extraksi fitur model topik modelling dengan classifier naïve bayes dan SVM

**Preprocessing**

In [11]:
import pandas as pd

# Baca dataset asli (bukan TF-IDF)
df = pd.read_csv("Berita.csv")  # ganti dengan nama file asli

# Jumlah total data
print("Jumlah Data:", len(df))

# Jumlah kolom
print("Jumlah Kolom:", len(df.columns))
print("Jumlah Kategori:", df['kategori'].nunique())
print("Daftar Kategori:", df['kategori'].unique())

print("\nJumlah Data per Kategori:")
print(df['kategori'].value_counts())
print(df.columns)

Jumlah Data: 1500
Jumlah Kolom: 6
Jumlah Kategori: 4
Daftar Kategori: ['Ekonomi' 'Olahraga' 'Nasional' 'Internasional']

Jumlah Data per Kategori:
kategori
Ekonomi          375
Olahraga         375
Nasional         375
Internasional    375
Name: count, dtype: int64
Index(['No', 'judul', 'berita', 'tanggal', 'kategori', 'link'], dtype='object')


membersihkan data berita / clean teks untuk kolom "berita"

In [10]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

nltk.download("stopwords")

# Load dataset hasil crawling
df = pd.read_csv("Berita.csv")

# Stopwords bahasa Indonesia
stopwords_indo = set(stopwords.words("indonesian"))

def clean_text(text):
    if pd.isna(text):  # kalau kosong
        return ""
    # lowercase
    text = text.lower()
    # hapus karakter non-huruf
    text = re.sub(r"[^a-zA-Záéíóúàèìòùüñç\s]", " ", text)
    # tokenisasi
    tokens = text.split()
    # hapus stopword
    tokens = [word for word in tokens if word not in stopwords_indo]
    return " ".join(tokens)

df["berita_clean"] = df["berita"].apply(clean_text)

# Simpan hasil ke file baru
df.to_csv("Berita_clean.csv", index=False, encoding="utf-8-sig")

# Tampilkan hasil 10 baris pertama
print(df[["berita", "berita_clean"]].head(10))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


                                              berita  \
0  Menteri Koordinator (Menko) Bidang Perekonomia...   
1  Dalam rangka memeriahkan hari jadi ke-50, PT S...   
2  Wacana Presiden Prabowo Subianto akan membentu...   
3  BPJS Ketenagakerjaan dan Kementerian Agama (Ke...   
4  Pemerintah akan segera membentuk Satuan Tugas ...   
5  Menko Bidang Infrastruktur dan Pembangunan Kew...   
6  Kepala Badan Gizi Nasional Dadan Hindayana men...   
7  Menteri Koordinator Bidang Pangan Zulkifli Has...   
8  Uji coba alias commissioning pembangkit listri...   
9  Anak crazy rich pengusaha sawit Kalimantan Sam...   

                                        berita_clean  
0  menteri koordinator menko bidang perekonomian ...  
1  rangka memeriahkan pt surabaya industrial esta...  
2  wacana presiden prabowo subianto membentuk mem...  
3  bpjs ketenagakerjaan kementerian agama kemenag...  
4  pemerintah membentuk satuan tugas pemutusan hu...  
5  menko bidang infrastruktur pembangunan kewilay... 

clean teks membersihkan symbol

In [14]:
import pandas as pd
import re
import string

df = pd.read_csv("Berita_clean.csv")

def remove_symbols(text):
    if pd.isna(text):
        return ""
    # hapus tanda baca standar (. , ! ? dll)
    text = text.translate(str.maketrans("", "", string.punctuation))
    # hapus karakter non-huruf (biar nggak ada simbol aneh)
    text = re.sub(r"[^a-zA-Záéíóúàèìòùüñç\s]", " ", text)
    # rapikan spasi
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Terapkan ke kolom hasil sebelumnya
df["berita_nosymbol"] = df["berita_clean"].apply(remove_symbols)

# Simpan hasil baru
df.to_csv("Berita_nosymbol.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df[["berita_clean", "berita_nosymbol"]].head(10))

                                        berita_clean  \
0  menteri koordinator menko bidang perekonomian ...   
1  rangka memeriahkan pt surabaya industrial esta...   
2  wacana presiden prabowo subianto membentuk mem...   
3  bpjs ketenagakerjaan kementerian agama kemenag...   
4  pemerintah membentuk satuan tugas pemutusan hu...   
5  menko bidang infrastruktur pembangunan kewilay...   
6  kepala badan gizi nasional dadan hindayana men...   
7  menteri koordinator bidang pangan zulkifli has...   
8  uji coba alias commissioning pembangkit listri...   
9  anak crazy rich pengusaha sawit kalimantan sam...   

                                     berita_nosymbol  
0  menteri koordinator menko bidang perekonomian ...  
1  rangka memeriahkan pt surabaya industrial esta...  
2  wacana presiden prabowo subianto membentuk mem...  
3  bpjs ketenagakerjaan kementerian agama kemenag...  
4  pemerintah membentuk satuan tugas pemutusan hu...  
5  menko bidang infrastruktur pembangunan kewilay... 

**Spell Cheker**

In [15]:
!pip install Sastrawi pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 55.8 MB/s eta 0:00:00


In [16]:
import pandas as pd
from spellchecker import SpellChecker

# Load dataset hasil sebelumnya
df = pd.read_csv("Berita_nosymbol.csv")

# Spell checker (default bahasa Inggris, kita isi dengan kosakata dataset)
spell = SpellChecker(language=None)
unique_words = set(" ".join(df["berita_nosymbol"].astype(str)).split())
spell.word_frequency.load_words(unique_words)

def spellcheck_text(text):
    if pd.isna(text):
        return ""
    tokens = text.split()
    corrected_tokens = []
    for token in tokens:
        if token not in spell:
            correction = spell.correction(token)
            corrected_tokens.append(correction if correction else token)
        else:
            corrected_tokens.append(token)
    return " ".join(corrected_tokens)

# Terapkan spellchecker ke abstrak
df["berita_spellchecked"] = df["berita_nosymbol"].apply(spellcheck_text)

# Simpan hasil
df.to_csv("Berita_spellchecked.csv", index=False, encoding="utf-8-sig")

print(df[["berita_nosymbol", "berita_spellchecked"]].head(10))

                                     berita_nosymbol  \
0  menteri koordinator menko bidang perekonomian ...   
1  rangka memeriahkan pt surabaya industrial esta...   
2  wacana presiden prabowo subianto membentuk mem...   
3  bpjs ketenagakerjaan kementerian agama kemenag...   
4  pemerintah membentuk satuan tugas pemutusan hu...   
5  menko bidang infrastruktur pembangunan kewilay...   
6  kepala badan gizi nasional dadan hindayana men...   
7  menteri koordinator bidang pangan zulkifli has...   
8  uji coba alias commissioning pembangkit listri...   
9  anak crazy rich pengusaha sawit kalimantan sam...   

                                 berita_spellchecked  
0  menteri koordinator menko bidang perekonomian ...  
1  rangka memeriahkan pt surabaya industrial esta...  
2  wacana presiden prabowo subianto membentuk mem...  
3  bpjs ketenagakerjaan kementerian agama kemenag...  
4  pemerintah membentuk satuan tugas pemutusan hu...  
5  menko bidang infrastruktur pembangunan kewilay... 

**Stemming**

In [17]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Load hasil spellchecker
df = pd.read_csv("Berita_spellchecked.csv")

# Buat stemmer bahasa Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming_text(text):
    if pd.isna(text):
        return ""
    return " ".join([stemmer.stem(word) for word in text.split()])

# Terapkan stemming
df["berita_stemmed"] = df["berita_spellchecked"].apply(stemming_text)

# Simpan hasil akhir
df.to_csv("Berita_stemmed.csv", index=False, encoding="utf-8-sig")

print(df[["berita_spellchecked", "berita_stemmed"]].head(10))

                                 berita_spellchecked  \
0  menteri koordinator menko bidang perekonomian ...   
1  rangka memeriahkan pt surabaya industrial esta...   
2  wacana presiden prabowo subianto membentuk mem...   
3  bpjs ketenagakerjaan kementerian agama kemenag...   
4  pemerintah membentuk satuan tugas pemutusan hu...   
5  menko bidang infrastruktur pembangunan kewilay...   
6  kepala badan gizi nasional dadan hindayana men...   
7  menteri koordinator bidang pangan zulkifli has...   
8  uji coba alias commissioning pembangkit listri...   
9  anak crazy rich pengusaha sawit kalimantan sam...   

                                      berita_stemmed  
0  menteri koordinator menko bidang ekonomi airla...  
1  rangka riah pt surabaya industrial estate rung...  
2  wacana presiden prabowo subianto bentuk bentuk...  
3  bpjs ketenagakerjaan menteri agama kemenag lin...  
4  perintah bentuk satu tugas putus hubung kerja ...  
5  menko bidang infrastruktur bangun wilayah agus... 

**Tokenizing**

In [18]:
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Download resource NLTK (sekali saja)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# Load hasil stemming
df = pd.read_csv("Berita_stemmed.csv")

# Stopwords bahasa Indonesia
stopwords_indo = set(stopwords.words("indonesian"))

def tokenize_text(text):
    if pd.isna(text):
        return []
    # tokenisasi
    tokens = nltk.word_tokenize(text)
    # buang stopword biar list lebih bersih
    tokens = [t for t in tokens if t.lower() not in stopwords_indo]
    return tokens

# Terapkan tokenisasi → hasil kolom berupa LIST
df["berita_tokens"] = df["berita_stemmed"].apply(tokenize_text)

# Simpan hasil
df.to_csv("Berita_tokenized.csv", index=False, encoding="utf-8-sig")

# Preview hasil
print(df["berita_tokens"].head(10))
print(type(df["berita_tokens"].iloc[0]))  # cek tipe data (harus list)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


0    [menteri, koordinator, menko, bidang, ekonomi,...
1    [rangka, riah, pt, surabaya, industrial, estat...
2    [wacana, presiden, prabowo, subianto, bentuk, ...
3    [bpjs, ketenagakerjaan, menteri, agama, kemena...
4    [perintah, bentuk, tugas, putus, hubung, kerja...
5    [menko, bidang, infrastruktur, bangun, wilayah...
6    [kepala, badan, gizi, nasional, dad, hindayana...
7    [menteri, koordinator, bidang, pangan, zulkifl...
8    [uji, coba, alias, commissioning, bangkit, lis...
9    [anak, crazy, rich, usaha, sawit, kalimantan, ...
Name: berita_tokens, dtype: object
<class 'list'>


In [19]:
#mengubah menjadi list
import pandas as pd
import ast

# Load hasil tokenisasi
df = pd.read_csv("Berita_tokenized.csv")

# Karena list disimpan sebagai string di CSV, perlu diubah balik ke list
df["berita_tokens"] = df["berita_tokens"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

# Gabungkan semua token ke dalam satu list besar
all_tokens = [token for tokens in df["berita_tokens"] for token in tokens]

print("Jumlah total token:", len(all_tokens))
print("50 token pertama:", all_tokens[:50])

Jumlah total token: 297672
50 token pertama: ['menteri', 'koordinator', 'menko', 'bidang', 'ekonomi', 'airlangga', 'hartarto', 'harap', 'upah', 'minimum', 'provinsi', 'ump', 'persen', 'tingkat', 'daya', 'beli', 'masyarakat', 'kelas', 'start', 'tunjang', 'daya', 'beli', 'masyarakat', 'kelas', 'upah', 'minimum', 'dongkrak', 'persen', 'airlangga', 'hadir', 'rapat', 'pimpin', 'nasional', 'rapimnas', 'kadin', 'jakarta', 'minggu', 'ketidakpastian', 'global', 'perintah', 'kuat', 'struktur', 'ekonomi', 'negeri', 'salah', 'struktur', 'giat', 'belanja', 'kelompok', 'masyarakat']


In [20]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 21.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatibl

Modeling Menggunakan file Berita_tokenized.csv dgn menggunakan ekstraksi fitur LDA

In [12]:
import pandas as pd
import ast
from gensim import corpora, models
import numpy as np

# === 1. Load dataset hasil tokenisasi ===
df = pd.read_csv("Berita_tokenized.csv")

# Pastikan kolom token dibaca sebagai list
df["berita_tokens"] = df["berita_tokens"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])

# === 2. Buat dictionary & corpus untuk LDA ===
dictionary = corpora.Dictionary(df["berita_tokens"])
corpus = [dictionary.doc2bow(text) for text in df["berita_tokens"]]

# === 3. Latih model LDA ===
lda_model = models.LdaModel(
    corpus,
    num_topics=10,
    id2word=dictionary,
    passes=10,
    random_state=42
)

# === 4. Tampilkan topik yang ditemukan ===
for idx, topic in lda_model.print_topics(num_topics=5):
    print(f"Topik {idx}: {topic}\n")

# === 5. Ubah distribusi topik tiap dokumen jadi fitur numerik ===
lda_features = [lda_model.get_document_topics(doc, minimum_probability=0) for doc in corpus]

# Konversi ke array numpy
X = np.array([[prob for _, prob in doc] for doc in lda_features])
print("Bentuk fitur LDA:", X.shape)

Topik 0: 0.036*"laut" + 0.028*"pagar" + 0.013*"menteri" + 0.010*"tangerang" + 0.009*"milik" + 0.008*"bangun" + 0.007*"nelayan" + 0.007*"perintah" + 0.007*"kkp" + 0.007*"ikan"

Topik 6: 0.017*"indonesia" + 0.014*"program" + 0.013*"makan" + 0.013*"rp" + 0.012*"gizi" + 0.012*"latih" + 0.009*"timnas" + 0.008*"gratis" + 0.008*"prabowo" + 0.007*"mbg"

Topik 7: 0.023*"main" + 0.018*"indonesia" + 0.013*"piala" + 0.012*"timnas" + 0.012*"gol" + 0.010*"menit" + 0.010*"menang" + 0.009*"laga" + 0.009*"tanding" + 0.009*"tim"

Topik 3: 0.013*"korban" + 0.010*"banjir" + 0.010*"duga" + 0.009*"polisi" + 0.007*"laku" + 0.007*"warga" + 0.007*"polda" + 0.007*"orang" + 0.007*"mobil" + 0.006*"rumah"

Topik 9: 0.006*"as" + 0.006*"harga" + 0.006*"masyarakat" + 0.006*"persen" + 0.005*"menteri" + 0.005*"jalan" + 0.005*"indonesia" + 0.005*"minyak" + 0.005*"tingkat" + 0.005*"negara"

Bentuk fitur LDA: (1500, 10)


Split data dan Klasifikasi

In [5]:
y = df["kategori"]

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# bagi data train-test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==== Naive Bayes ====
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("=== HASIL NAIVE BAYES ===")
print(classification_report(y_test, y_pred_nb))

# ==== SVM ====
svm = SVC(kernel='linear', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("\n=== HASIL SVM ===")
print(classification_report(y_test, y_pred_svm))

=== HASIL NAIVE BAYES ===
               precision    recall  f1-score   support

      Ekonomi       0.46      0.16      0.24        73
Internasional       0.62      0.79      0.69        84
     Nasional       0.62      0.84      0.72        70
     Olahraga       0.97      0.96      0.97        73

     accuracy                           0.69       300
    macro avg       0.67      0.69      0.65       300
 weighted avg       0.67      0.69      0.65       300


=== HASIL SVM ===
               precision    recall  f1-score   support

      Ekonomi       0.50      0.08      0.14        73
Internasional       0.61      0.83      0.70        84
     Nasional       0.56      0.83      0.67        70
     Olahraga       0.99      0.95      0.97        73

     accuracy                           0.68       300
    macro avg       0.66      0.67      0.62       300
 weighted avg       0.66      0.68      0.62       300

